# 00 — Bootstrap test

Smoke test que cada miembro ejecuta **una vez** al estrenar entorno (Colab o local). Verifica:

1. Clone del repo (Colab) o presencia local del código.
2. Instalación de `requirements.txt`.
3. `configs/local.yaml` presente y bien formado.
4. Acceso al dataset (montaje Drive + cache local + symlink).
5. Imports de todos los módulos `src/`.
6. Carga de configs con herencia `extends:`.
7. Funciones puras (`anonymize`, `clean_text`) sobre un ejemplo.

Si esta notebook termina sin errores, el resto del pipeline funciona en este entorno.

## 1. Bootstrap código + dependencias

In [ ]:
REPO_URL = 'https://github.com/elvinsomon/pln-poc.git'

import os, sys, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir('/content/pln-poc'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/pln-poc'], check=True)
    os.chdir('/content/pln-poc')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('IN_COLAB    :', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)

## 2. Imports + seed

In [ ]:
from src.utils.colab import setup_environment, in_colab, bootstrap_dataset, load_local_config
from src.utils.config import load_config
from src.data.anonymize import anonymize
from src.data.clean import clean_text
from src.data.splits import prepare_splits, load_splits
from src.models.majority import train_majority
from src.models.tfidf_svm import build_pipeline
from src.evaluation.metrics import compute_metrics

import sklearn, pandas, numpy
print('python  :', sys.version.split()[0])
print('sklearn :', sklearn.__version__)
print('pandas  :', pandas.__version__)
print('numpy   :', numpy.__version__)

root = setup_environment(seed=42, project_root=PROJECT_ROOT)
print('\nroot  :', root)
print('Colab :', in_colab())

## 3. Configs (`extends:` + `local.yaml`)

`configs/local.yaml` es **personal** (no versionado). Si falla este paso, sigue las instrucciones del error: `cp configs/local.example.yaml configs/local.yaml` y ajusta `dataset_on_drive` a tu ruta de Drive.

In [ ]:
for name in ('base.yaml', 'data.yaml', 'tfidf_svm.yaml'):
    print(f'{name:18s} -> {list(load_config(name).keys())}')

cfg = load_config('tfidf_svm.yaml')
assert cfg['seed'] == 42
assert cfg['classes'] == ['bug', 'feature', 'question']

if IN_COLAB:
    local = load_local_config()
    print('\nlocal.yaml ->', list(local.keys()))
    print('dataset_on_drive:', local['dataset_on_drive'])

print('\nconfigs OK')

## 4. Smoke de funciones puras

In [ ]:
sample = 'Contact me at user@example.com or @octocat at https://github.com/x.\n```py\nraise NullPointerException\n```\n**bold** [link](http://x.io) <p>html</p>'
print('anon :', anonymize(sample))
print('clean:', clean_text(anonymize(sample), max_chars=200))

## 5. Dataset accesible

En Colab: monta Drive (si no estaba), copia el CSV a cache local si `copy_to_local: true`, y enlaza al path esperado por los configs. En local: solo verifica que el CSV está en `data/raw/`.

In [ ]:
from pathlib import Path
csv_path = bootstrap_dataset(load_config('base.yaml'))
print('CSV   :', csv_path)
print('size  :', f'{csv_path.stat().st_size / 1e9:.2f} GB')